# Bayesian Posterior Diagnostics And Targeted Scale Models

This notebook diagnoses where the tuned Bayesian linear model is still uncertain or biased, then tests a small set of targeted heteroskedastic Student-t scale models. The goal is to decide whether Bayesian linear scale structure is worth extending before moving to Hilbert-space Gaussian process residual modeling.


## 1. Setup

In [1]:
"Configure imports, paths, JAX, and deterministic settings."

from pathlib import Path
import sys
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import SVI, Trace_ELBO, Predictive
from numpyro.infer.autoguide import AutoDelta, AutoNormal
from numpyro.optim import Adam

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = next(parent for parent in Path.cwd().parents if (parent / "src").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation.validation import group_well_folds
from src.io.loaders import discover_wells, load_horizontal_well
from src.utils.run_bundle import RunBundle

warnings.filterwarnings("ignore", category=RuntimeWarning)

RUN_TAG = "bayesian_posterior_diagnostics_scale"
RUN_DATE = "2026-06-22"
SEED = 42
N_SPLITS = 3
MAX_TARGET_ROWS_PER_WELL = 140
MAX_TRAIN_ROWS_PER_FOLD = 60000
SVI_STEPS = 1000
LEARNING_RATE = 0.02
POSTERIOR_SAMPLES = 160
RESIDUAL_SAMPLE_PER_FOLD = 2200

DATA_DIR = ROOT / "data" / "raw"
TRAIN_DIR = DATA_DIR / "train"
ARTIFACT_DIR = ROOT / "artifacts"
TABLE_DIR = ARTIFACT_DIR / "tables"
FIGURE_DIR = ARTIFACT_DIR / "figures"
REPORT_DIR = ARTIFACT_DIR / "reports"

for directory in [TABLE_DIR, FIGURE_DIR, REPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
numpyro.set_platform("cpu")
plt.style.use("seaborn-v0_8-whitegrid")
print({"jax": jax.__version__, "numpyro": numpyro.__version__, "devices": [str(device) for device in jax.devices()]})


C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'jax': '0.10.2', 'numpyro': '0.21.0', 'devices': ['cpu:0']}


## 2. Slope-Core Feature Frame

The feature frame matches the previous Bayesian tuning notebook so that any change in RMSE or coverage is attributable to posterior diagnostics and targeted scale modeling, not a silent feature-builder change.


In [2]:
"Define notebook-local slope-core feature construction helpers."


def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce").astype(float)


def deterministic_sample_positions(mask, max_rows):
    positions = np.flatnonzero(mask.to_numpy())
    if len(positions) <= max_rows:
        return positions
    return positions[np.linspace(0, len(positions) - 1, max_rows).astype(int)]


def deterministic_row_cap(frame, max_rows):
    if len(frame) <= max_rows:
        return frame
    positions = np.linspace(0, len(frame) - 1, max_rows).astype(int)
    return frame.iloc[positions].copy()


def window_slope(md, values, end_pos, window):
    start_pos = max(0, end_pos - window + 1)
    x0 = float(md.iloc[start_pos])
    x1 = float(md.iloc[end_pos])
    y0 = float(values.iloc[start_pos])
    y1 = float(values.iloc[end_pos])
    if not np.isfinite([x0, x1, y0, y1]).all() or abs(x1 - x0) < 1e-9:
        return np.nan
    return (y1 - y0) / (x1 - x0)


def build_slope_core_features_for_well(well, directory, max_rows):
    horizontal = load_horizontal_well(directory / f"{well}__horizontal_well.csv")
    target_mask = horizontal["TVT_input"].isna() & horizontal["TVT"].notna()
    known_mask = horizontal["TVT_input"].notna() & horizontal["TVT"].notna()
    if int(target_mask.sum()) == 0 or int(known_mask.sum()) == 0:
        return pd.DataFrame()

    tvt_input = safe_numeric(horizontal["TVT_input"])
    md = safe_numeric(horizontal["MD"])
    x = safe_numeric(horizontal["X"])
    y = safe_numeric(horizontal["Y"])
    z = safe_numeric(horizontal["Z"])

    known_positions = np.flatnonzero(known_mask.to_numpy())
    last_known_pos = int(known_positions[-1])
    last_known_tvt = float(tvt_input.iloc[last_known_pos])
    last_known_md = float(md.iloc[last_known_pos])
    last_known_z = float(z.iloc[last_known_pos])

    slope_windows = [10, 25, 50, 100]
    tvt_slopes = {f"tvt_slope_last_{window}": window_slope(md, tvt_input, last_known_pos, window) for window in slope_windows}
    z_slopes = {f"z_slope_last_{window}": window_slope(md, z, last_known_pos, window) for window in slope_windows}
    horizontal_step = np.sqrt(x.diff().fillna(0.0) ** 2 + y.diff().fillna(0.0) ** 2)
    horizontal_dist = horizontal_step.cumsum()

    target_positions = deterministic_sample_positions(target_mask, max_rows)
    all_target_positions = np.flatnonzero(target_mask.to_numpy())
    eval_rank = {int(pos): rank for rank, pos in enumerate(all_target_positions)}
    eval_denominator = max(len(all_target_positions) - 1, 1)

    rows = []
    for pos in target_positions:
        distance_md = float(md.iloc[pos] - last_known_md)
        row = {
            "wellname": well,
            "row_idx": int(horizontal.index[pos]),
            "md_norm": float((md.iloc[pos] - md.min()) / (md.max() - md.min() + 1e-9)),
            "distance_from_boundary_md": distance_md,
            "eval_progress": float(eval_rank[int(pos)] / eval_denominator),
            "last_known_tvt": last_known_tvt,
            "dZ_from_boundary": float(z.iloc[pos] - last_known_z),
            "horizontal_dist_from_boundary": float(horizontal_dist.iloc[pos] - horizontal_dist.iloc[last_known_pos]),
            "TVT": float(horizontal["TVT"].iloc[pos]),
            "endpoint_hold_pred": last_known_tvt,
        }
        row["delta_tvt_from_boundary"] = row["TVT"] - last_known_tvt
        for name, value in tvt_slopes.items():
            row[name] = float(value) if np.isfinite(value) else np.nan
            row[f"{name}_projected_delta"] = row[name] * distance_md if np.isfinite(row[name]) else np.nan
        for name, value in z_slopes.items():
            row[name] = float(value) if np.isfinite(value) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


slope_features = ["tvt_slope_last_10", "tvt_slope_last_25", "tvt_slope_last_50", "tvt_slope_last_100", "z_slope_last_25", "z_slope_last_50", "z_slope_last_100"]
slope_projection_features = ["tvt_slope_last_10_projected_delta", "tvt_slope_last_25_projected_delta", "tvt_slope_last_50_projected_delta", "tvt_slope_last_100_projected_delta"]
feature_columns = ["distance_from_boundary_md", "eval_progress", "md_norm", "dZ_from_boundary", "horizontal_dist_from_boundary"] + slope_features + slope_projection_features + ["last_known_tvt"]

print({"n_mean_features": len(feature_columns)})


{'n_mean_features': 17}


In [3]:
"Build the sampled target-zone modeling frame and targeted scale features."

start = time.time()
train_wells = discover_wells(TRAIN_DIR)
frames = [build_slope_core_features_for_well(well, TRAIN_DIR, MAX_TARGET_ROWS_PER_WELL) for well in train_wells]
feature_frame = pd.concat([frame for frame in frames if not frame.empty], ignore_index=True)
build_seconds = time.time() - start

feature_frame["distance_log1p"] = np.log1p(feature_frame["distance_from_boundary_md"].clip(lower=0.0))
feature_frame["horizontal_log1p"] = np.log1p(feature_frame["horizontal_dist_from_boundary"].clip(lower=0.0))
feature_frame["abs_dZ_from_boundary"] = feature_frame["dZ_from_boundary"].abs()
feature_frame["abs_tvt_slope_last_10"] = feature_frame["tvt_slope_last_10"].abs()
feature_frame["abs_tvt_slope_last_50"] = feature_frame["tvt_slope_last_50"].abs()
feature_frame["abs_projected_delta_10"] = feature_frame["tvt_slope_last_10_projected_delta"].abs()
feature_frame["progress_centered"] = feature_frame["eval_progress"] - 0.5
feature_frame["progress_squared"] = feature_frame["progress_centered"] ** 2
feature_frame["distance_progress"] = feature_frame["distance_log1p"] * feature_frame["eval_progress"]

print({"wells": len(train_wells), "sample_rows": len(feature_frame), "build_seconds": round(build_seconds, 2)})
feature_frame.head()


12:51:01 | INFO     | src.io.loaders | Discovered 773 wells in C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\data\raw\train


{'wells': 773, 'sample_rows': 108220, 'build_seconds': 45.71}


,wellname,row_idx,md_norm,distance_from_boundary_md,eval_progress,last_known_tvt,dZ_from_boundary,horizontal_dist_from_boundary,TVT,endpoint_hold_pred,...,z_slope_last_100,distance_log1p,horizontal_log1p,abs_dZ_from_boundary,abs_tvt_slope_last_10,abs_tvt_slope_last_50,abs_projected_delta_10,progress_centered,progress_squared,distance_progress
0,000d7d20,1442,0.273261,1.0,0.000000,11747.37,0.01,1.000450,11747.38,11747.37,...,0.004343,0.693147,0.693372,0.01,0.011111,0.016327,0.011111,-0.500000,0.250000,0.000000
1,000d7d20,1469,0.278378,28.0,0.007040,11747.37,0.39,27.992356,11747.52,11747.37,...,0.004343,3.367296,3.367032,0.39,0.011111,0.016327,0.311111,-0.492960,0.243009,0.023707
2,000d7d20,1497,0.283684,56.0,0.014342,11747.37,1.02,55.983960,11747.46,11747.37,...,0.004343,4.043051,4.042770,1.02,0.011111,0.016327,0.622222,-0.485658,0.235864,0.057984
3,000d7d20,1524,0.288800,83.0,0.021382,11747.37,1.80,82.975967,11748.05,11747.37,...,0.004343,4.430817,4.430531,1.80,0.011111,0.016327,0.922222,-0.478618,0.229075,0.094740
4,000d7d20,1552,0.294106,111.0,0.028683,11747.37,2.68,110.962778,11748.66,11747.37,...,0.004343,4.718499,4.718166,2.68,0.011111,0.016327,1.233333,-0.471317,0.222140,0.135342


## 3. Model Definitions

All candidates keep the same fixed-tail Student-t likelihood and linear mean structure. The only intentional change is the row-level scale model.


In [4]:
"Define fixed and targeted heteroskedastic NumPyro models plus fold helpers."


def linear_mu(X):
    n_features = X.shape[1]
    intercept = numpyro.sample("intercept", dist.Normal(0.0, 1.5))
    beta = numpyro.sample("beta", dist.Normal(0.0, 2.0).expand([n_features]).to_event(1))
    return intercept + jnp.dot(X, beta)


def fixed_student_t_model(X, X_sigma, y=None):
    mu = linear_mu(X)
    sigma = numpyro.sample("sigma", dist.HalfNormal(1.5))
    sigma_row = jnp.ones(X.shape[0]) * sigma
    numpyro.deterministic("mu", mu)
    numpyro.deterministic("sigma_row", sigma_row)
    numpyro.sample("obs", dist.StudentT(3.0, mu, sigma_row), obs=y)


def hetero_student_t_model(X, X_sigma, y=None):
    mu = linear_mu(X)
    n_sigma_features = X_sigma.shape[1]
    sigma_intercept = numpyro.sample("sigma_intercept", dist.Normal(-0.2, 0.8))
    gamma = numpyro.sample("gamma", dist.Normal(0.0, 0.35).expand([n_sigma_features]).to_event(1))
    sigma_row = jax.nn.softplus(sigma_intercept + jnp.dot(X_sigma, gamma)) + 1e-4
    numpyro.deterministic("mu", mu)
    numpyro.deterministic("sigma_row", sigma_row)
    numpyro.sample("obs", dist.StudentT(3.0, mu, sigma_row), obs=y)


scale_candidate_specs = [
    {"candidate": "fixed_t3_autodelta_reference", "model": fixed_student_t_model, "guide": "autodelta", "scale_family": "fixed", "sigma_columns": []},
    {"candidate": "hetero_boundary_progress_autodelta", "model": hetero_student_t_model, "guide": "autodelta", "scale_family": "boundary_progress", "sigma_columns": ["distance_log1p", "eval_progress", "progress_squared", "distance_progress"]},
    {"candidate": "hetero_trajectory_autodelta", "model": hetero_student_t_model, "guide": "autodelta", "scale_family": "trajectory", "sigma_columns": ["distance_log1p", "horizontal_log1p", "abs_dZ_from_boundary", "md_norm"]},
    {"candidate": "hetero_slope_instability_autodelta", "model": hetero_student_t_model, "guide": "autodelta", "scale_family": "slope_instability", "sigma_columns": ["distance_log1p", "eval_progress", "abs_tvt_slope_last_10", "abs_tvt_slope_last_50", "abs_projected_delta_10"]},
    {"candidate": "hetero_boundary_progress_autonormal", "model": hetero_student_t_model, "guide": "autonormal", "scale_family": "boundary_progress", "sigma_columns": ["distance_log1p", "eval_progress", "progress_squared", "distance_progress"]},
]


def make_guide(spec):
    if spec["guide"] == "autonormal":
        return AutoNormal(spec["model"])
    return AutoDelta(spec["model"])


def fit_preprocessor(train_frame, columns):
    if len(columns) == 0:
        return pd.Series(dtype=float), pd.Series(dtype=float), pd.Series(dtype=float)
    medians = train_frame[columns].median(numeric_only=True)
    filled = train_frame[columns].fillna(medians)
    means = filled.mean()
    stds = filled.std().replace(0.0, 1.0).fillna(1.0)
    return medians, means, stds


def apply_preprocessor(frame, columns, medians, means, stds):
    if len(columns) == 0:
        return np.zeros((len(frame), 1), dtype=float)
    filled = frame[columns].fillna(medians)
    return ((filled - means) / stds).to_numpy(float)


def prepare_fold_arrays(train_frame, valid_frame, sigma_columns):
    x_medians, x_means, x_stds = fit_preprocessor(train_frame, feature_columns)
    s_medians, s_means, s_stds = fit_preprocessor(train_frame, sigma_columns)
    X_train = apply_preprocessor(train_frame, feature_columns, x_medians, x_means, x_stds)
    X_valid = apply_preprocessor(valid_frame, feature_columns, x_medians, x_means, x_stds)
    Xs_train = apply_preprocessor(train_frame, sigma_columns, s_medians, s_means, s_stds)
    Xs_valid = apply_preprocessor(valid_frame, sigma_columns, s_medians, s_means, s_stds)
    y_train_raw = train_frame["delta_tvt_from_boundary"].to_numpy(float)
    y_mean = float(np.mean(y_train_raw))
    y_std = float(np.std(y_train_raw) + 1e-9)
    y_train = (y_train_raw - y_mean) / y_std
    return {"X_train": jnp.asarray(X_train), "X_valid": jnp.asarray(X_valid), "Xs_train": jnp.asarray(Xs_train), "Xs_valid": jnp.asarray(Xs_valid), "y_train": jnp.asarray(y_train), "y_mean": y_mean, "y_std": y_std}


def metric_row(y_true, pred_tvt):
    return {"rmse": float(np.sqrt(mean_squared_error(y_true, pred_tvt))), "mae": float(mean_absolute_error(y_true, pred_tvt)), "r2": float(r2_score(y_true, pred_tvt))}


def interval_arrays(draws):
    return {"q10": np.quantile(draws, 0.10, axis=0), "q90": np.quantile(draws, 0.90, axis=0), "q025": np.quantile(draws, 0.025, axis=0), "q975": np.quantile(draws, 0.975, axis=0)}


def interval_metrics(y_true, interval_values):
    q10 = interval_values["q10"]
    q90 = interval_values["q90"]
    q025 = interval_values["q025"]
    q975 = interval_values["q975"]
    return {"coverage_80": float(((y_true >= q10) & (y_true <= q90)).mean()), "coverage_95": float(((y_true >= q025) & (y_true <= q975)).mean()), "width_80": float(np.mean(q90 - q10)), "width_95": float(np.mean(q975 - q025))}


def bin_labels(values, n_bins=6):
    series = pd.Series(values).replace([np.inf, -np.inf], np.nan)
    if series.nunique(dropna=True) < 2:
        return pd.Series(["all"] * len(series))
    ranked = series.rank(method="first")
    bins = pd.qcut(ranked, q=min(n_bins, int(series.nunique(dropna=True))), duplicates="drop")
    return bins.astype(str)


pd.DataFrame([{k: spec[k] for k in ["candidate", "guide", "scale_family", "sigma_columns"]} for spec in scale_candidate_specs])


,candidate,guide,scale_family,sigma_columns
0,fixed_t3_autodelta_reference,autodelta,fixed,[]
1,hetero_boundary_progress_autodelta,autodelta,boundary_progress,"[distance_log1p, eval_progress, progress_squar..."
2,hetero_trajectory_autodelta,autodelta,trajectory,"[distance_log1p, horizontal_log1p, abs_dZ_from..."
3,hetero_slope_instability_autodelta,autodelta,slope_instability,"[distance_log1p, eval_progress, abs_tvt_slope_..."
4,hetero_boundary_progress_autonormal,autonormal,boundary_progress,"[distance_log1p, eval_progress, progress_squar..."


## 4. Grouped CV And Posterior Predictive Diagnostics

In [5]:
"Run grouped well-fold validation and collect coverage, residual, and scale diagnostics."

fit_start = time.time()
fold_rows = []
residual_rows = []
posterior_rows = []
bin_rows = []
well_rows = []
wells = sorted(feature_frame["wellname"].unique())
folds = group_well_folds(wells, n_splits=N_SPLITS)
diagnostic_features = ["distance_from_boundary_md", "eval_progress", "horizontal_dist_from_boundary", "abs_tvt_slope_last_10", "abs_projected_delta_10"]

for fold_idx, (train_fold_wells, valid_fold_wells) in enumerate(folds, start=1):
    train_frame = feature_frame.loc[feature_frame["wellname"].isin(train_fold_wells)].copy()
    valid_frame = feature_frame.loc[feature_frame["wellname"].isin(valid_fold_wells)].copy()
    train_frame = deterministic_row_cap(train_frame, MAX_TRAIN_ROWS_PER_FOLD)
    y_true_tvt = valid_frame["TVT"].to_numpy(float)
    endpoint_pred = valid_frame["endpoint_hold_pred"].to_numpy(float)
    fold_rows.append({"fold": fold_idx, "candidate": "endpoint_hold", "guide": "baseline", "scale_family": "baseline", "n_sigma_features": 0, "n_train_rows": 0, "n_valid_rows": len(valid_frame), "final_loss": np.nan, "coverage_80": np.nan, "coverage_95": np.nan, "width_80": np.nan, "width_95": np.nan, "posterior_sd_mean": np.nan, "sd_abs_error_corr": np.nan, **metric_row(y_true_tvt, endpoint_pred)})

    for spec_idx, spec in enumerate(scale_candidate_specs):
        arrays = prepare_fold_arrays(train_frame, valid_frame, spec["sigma_columns"])
        rng_key = jax.random.PRNGKey(SEED + fold_idx * 100 + spec_idx)
        guide = make_guide(spec)
        svi = SVI(spec["model"], guide, Adam(LEARNING_RATE), Trace_ELBO())
        svi_result = svi.run(rng_key, SVI_STEPS, arrays["X_train"], arrays["Xs_train"], arrays["y_train"], progress_bar=False)
        params = svi_result.params
        final_loss = float(svi_result.losses[-1])
        pred_key = jax.random.PRNGKey(SEED + fold_idx * 1000 + spec_idx)
        predictive = Predictive(spec["model"], guide=guide, params=params, num_samples=POSTERIOR_SAMPLES, return_sites=("mu", "obs", "sigma_row"))
        pred = predictive(pred_key, arrays["X_valid"], arrays["Xs_valid"], None)
        mu_draws_delta = np.asarray(pred["mu"]) * arrays["y_std"] + arrays["y_mean"]
        obs_draws_delta = np.asarray(pred["obs"]) * arrays["y_std"] + arrays["y_mean"]
        sigma_draws = np.asarray(pred["sigma_row"]) * arrays["y_std"]
        last_known = valid_frame["last_known_tvt"].to_numpy(float)[None, :]
        tvt_mu_draws = mu_draws_delta + last_known
        tvt_obs_draws = obs_draws_delta + last_known
        pred_tvt = tvt_mu_draws.mean(axis=0)
        residual = y_true_tvt - pred_tvt
        abs_error = np.abs(residual)
        posterior_sd = tvt_obs_draws.std(axis=0)
        intervals = interval_arrays(tvt_obs_draws)
        interval_summary = interval_metrics(y_true_tvt, intervals)
        sd_abs_error_corr = float(pd.Series(posterior_sd).corr(pd.Series(abs_error), method="spearman"))
        fold_rows.append({"fold": fold_idx, "candidate": spec["candidate"], "guide": spec["guide"], "scale_family": spec["scale_family"], "n_sigma_features": len(spec["sigma_columns"]), "n_train_rows": len(train_frame), "n_valid_rows": len(valid_frame), "final_loss": final_loss, "posterior_sd_mean": float(np.mean(posterior_sd)), "sd_abs_error_corr": sd_abs_error_corr, **metric_row(y_true_tvt, pred_tvt), **interval_summary})
        posterior_rows.append({"fold": fold_idx, "candidate": spec["candidate"], "guide": spec["guide"], "scale_family": spec["scale_family"], "final_loss": final_loss, "mean_sigma_row": float(np.mean(sigma_draws)), "sd_sigma_row": float(np.std(sigma_draws)), "min_sigma_row": float(np.min(sigma_draws)), "max_sigma_row": float(np.max(sigma_draws)), "sd_abs_error_spearman": sd_abs_error_corr})

        full_diag = valid_frame[["wellname", "row_idx"] + diagnostic_features].copy()
        full_diag["y_true"] = y_true_tvt
        full_diag["y_pred"] = pred_tvt
        full_diag["residual"] = residual
        full_diag["abs_error"] = abs_error
        full_diag["posterior_sd"] = posterior_sd
        full_diag["inside_80"] = (y_true_tvt >= intervals["q10"]) & (y_true_tvt <= intervals["q90"])
        full_diag["inside_95"] = (y_true_tvt >= intervals["q025"]) & (y_true_tvt <= intervals["q975"])
        full_diag["width_80"] = intervals["q90"] - intervals["q10"]

        for feature_name in diagnostic_features:
            full_diag["diagnostic_bin"] = bin_labels(full_diag[feature_name])
            grouped = full_diag.groupby("diagnostic_bin", observed=False)
            for bin_name, group in grouped:
                bin_rows.append({"fold": fold_idx, "candidate": spec["candidate"], "feature": feature_name, "bin": str(bin_name), "n_rows": int(len(group)), "feature_min": float(group[feature_name].min()), "feature_max": float(group[feature_name].max()), "rmse": float(np.sqrt(mean_squared_error(group["y_true"], group["y_pred"]))), "mean_residual": float(group["residual"].mean()), "mean_abs_error": float(group["abs_error"].mean()), "coverage_80": float(group["inside_80"].mean()), "coverage_95": float(group["inside_95"].mean()), "width_80": float(group["width_80"].mean()), "posterior_sd_mean": float(group["posterior_sd"].mean())})

        for well, group in full_diag.groupby("wellname"):
            well_rows.append({"fold": fold_idx, "candidate": spec["candidate"], "wellname": well, "n_rows": int(len(group)), "rmse": float(np.sqrt(mean_squared_error(group["y_true"], group["y_pred"]))), "mean_residual": float(group["residual"].mean()), "mean_abs_error": float(group["abs_error"].mean()), "coverage_80": float(group["inside_80"].mean()), "posterior_sd_mean": float(group["posterior_sd"].mean())})

        sample_idx = np.linspace(0, len(valid_frame) - 1, min(len(valid_frame), RESIDUAL_SAMPLE_PER_FOLD)).astype(int)
        residual_rows.append(full_diag.iloc[sample_idx].assign(candidate=spec["candidate"], fold=fold_idx))

fit_seconds = time.time() - fit_start
fold_metrics = pd.DataFrame(fold_rows)
summary = (fold_metrics.groupby(["candidate", "guide", "scale_family"], as_index=False).agg(rmse_mean=("rmse", "mean"), rmse_std=("rmse", "std"), mae_mean=("mae", "mean"), r2_mean=("r2", "mean"), coverage_80_mean=("coverage_80", "mean"), coverage_95_mean=("coverage_95", "mean"), width_80_mean=("width_80", "mean"), width_95_mean=("width_95", "mean"), posterior_sd_mean=("posterior_sd_mean", "mean"), sd_abs_error_corr_mean=("sd_abs_error_corr", "mean"), n_valid_rows=("n_valid_rows", "sum"), n_train_rows_mean=("n_train_rows", "mean"), final_loss_mean=("final_loss", "mean")).sort_values("rmse_mean"))
residual_sample = pd.concat(residual_rows, ignore_index=True)
posterior_diagnostics = pd.DataFrame(posterior_rows)
bin_diagnostics = pd.DataFrame(bin_rows)
well_diagnostics = pd.DataFrame(well_rows)

print({"fit_seconds": round(fit_seconds, 2), "fold_rows": len(fold_metrics), "bin_rows": len(bin_diagnostics), "well_rows": len(well_diagnostics), "residual_rows": len(residual_sample)})
summary


{'fit_seconds': 162.48, 'fold_rows': 18, 'bin_rows': 450, 'well_rows': 3865, 'residual_rows': 33000}


,candidate,guide,scale_family,rmse_mean,rmse_std,mae_mean,r2_mean,coverage_80_mean,coverage_95_mean,width_80_mean,width_95_mean,posterior_sd_mean,sd_abs_error_corr_mean,n_valid_rows,n_train_rows_mean,final_loss_mean
2,hetero_boundary_progress_autodelta,autodelta,boundary_progress,14.862579,0.501949,10.415761,0.999424,0.780177,0.944615,31.945421,60.686808,16.457313,0.314684,108220,60000.0,72025.921875
1,fixed_t3_autodelta_reference,autodelta,fixed,14.912892,0.575342,10.443034,0.999420,0.770290,0.936510,31.316723,59.479067,16.149630,0.004813,108220,60000.0,77617.520833
3,hetero_boundary_progress_autonormal,autonormal,boundary_progress,14.922672,0.473479,10.466566,0.999419,0.781487,0.945013,32.133260,61.036104,16.560436,0.309106,108220,60000.0,72720.541667
5,hetero_trajectory_autodelta,autodelta,trajectory,14.957500,0.571444,10.514815,0.999416,0.782255,0.945335,32.275113,61.312224,16.627540,0.312390,108220,60000.0,72346.333333
4,hetero_slope_instability_autodelta,autodelta,slope_instability,15.072983,0.587857,10.644323,0.999407,0.786014,0.949439,33.263755,63.194209,17.166131,0.322129,108220,60000.0,71646.934896
0,endpoint_hold,baseline,baseline,15.578175,0.651555,10.905266,0.999367,NaN,NaN,NaN,NaN,NaN,NaN,108220,0.0,NaN


## 5. Diagnostic Tables And Figures

In [6]:
"Export scale metrics, posterior diagnostics, residual diagnostics, and figures."

fold_metrics_path = TABLE_DIR / f"model__bayesian_posterior_scale_fold_metrics__{RUN_DATE}.csv"
summary_path = TABLE_DIR / f"model__bayesian_posterior_scale_summary__{RUN_DATE}.csv"
posterior_path = TABLE_DIR / f"model__bayesian_posterior_scale_posterior_diagnostics__{RUN_DATE}.csv"
residual_path = TABLE_DIR / f"model__bayesian_posterior_scale_residual_sample__{RUN_DATE}.csv"
bin_path = TABLE_DIR / f"model__bayesian_posterior_scale_bin_diagnostics__{RUN_DATE}.csv"
well_path = TABLE_DIR / f"model__bayesian_posterior_scale_well_diagnostics__{RUN_DATE}.csv"
fold_metrics.to_csv(fold_metrics_path, index=False)
summary.to_csv(summary_path, index=False)
posterior_diagnostics.to_csv(posterior_path, index=False)
residual_sample.to_csv(residual_path, index=False)
bin_diagnostics.to_csv(bin_path, index=False)
well_diagnostics.to_csv(well_path, index=False)

fig_paths = {}
plot_summary = summary.sort_values("rmse_mean", ascending=True)
fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#4C78A8" if candidate != "endpoint_hold" else "#F58518" for candidate in plot_summary["candidate"]]
ax.barh(plot_summary["candidate"], plot_summary["rmse_mean"], color=colors)
ax.set_title("Bayesian Targeted Scale RMSE")
ax.set_xlabel("grouped target-zone RMSE")
ax.set_ylabel("")
fig.tight_layout()
fig_paths["rmse_comparison"] = FIGURE_DIR / f"model__bayesian_posterior_scale_rmse_comparison__{RUN_DATE}.png"
fig.savefig(fig_paths["rmse_comparison"], dpi=160)
plt.close(fig)

best_candidate = str(summary.loc[summary["candidate"] != "endpoint_hold"].iloc[0]["candidate"])
best_bins = bin_diagnostics.loc[(bin_diagnostics["candidate"] == best_candidate) & (bin_diagnostics["feature"] == "distance_from_boundary_md")].copy()
best_bins = best_bins.sort_values(["fold", "feature_min"])
fold_avg_bins = best_bins.groupby("bin", as_index=False).agg(feature_min=("feature_min", "mean"), feature_max=("feature_max", "mean"), coverage_80=("coverage_80", "mean"), rmse=("rmse", "mean"), width_80=("width_80", "mean")).sort_values("feature_min")
fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(range(len(fold_avg_bins)), fold_avg_bins["coverage_80"], marker="o", color="#54A24B", label="80% coverage")
ax1.axhline(0.80, color="black", linestyle="--", linewidth=1)
ax1.set_ylabel("80% coverage")
ax1.set_xlabel("boundary-distance quantile")
ax2 = ax1.twinx()
ax2.plot(range(len(fold_avg_bins)), fold_avg_bins["rmse"], marker="s", color="#E45756", label="RMSE")
ax2.set_ylabel("RMSE")
ax1.set_title(f"Coverage And RMSE By Boundary Distance: {best_candidate}")
fig.tight_layout()
fig_paths["coverage_rmse_by_boundary"] = FIGURE_DIR / f"model__bayesian_posterior_scale_coverage_rmse_by_boundary__{RUN_DATE}.png"
fig.savefig(fig_paths["coverage_rmse_by_boundary"], dpi=160)
plt.close(fig)

best_residuals = residual_sample.loc[residual_sample["candidate"] == best_candidate].copy()
best_residuals["progress_bin"] = bin_labels(best_residuals["eval_progress"])
progress_diag = best_residuals.groupby("progress_bin", observed=False).agg(eval_progress=("eval_progress", "mean"), mean_residual=("residual", "mean"), mean_abs_error=("abs_error", "mean")).reset_index().sort_values("eval_progress")
fig, ax = plt.subplots(figsize=(8, 5))
ax.axhline(0.0, color="black", linewidth=1)
ax.plot(progress_diag["eval_progress"], progress_diag["mean_residual"], marker="o", color="#4C78A8")
ax.set_title(f"Mean Residual By Eval Progress: {best_candidate}")
ax.set_xlabel("mean eval progress by bin")
ax.set_ylabel("mean residual")
fig.tight_layout()
fig_paths["residual_bias_by_progress"] = FIGURE_DIR / f"model__bayesian_posterior_scale_residual_bias_by_progress__{RUN_DATE}.png"
fig.savefig(fig_paths["residual_bias_by_progress"], dpi=160)
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(best_residuals["posterior_sd"], best_residuals["abs_error"], s=7, alpha=0.35, color="#B279A2")
ax.set_title(f"Posterior SD vs Absolute Error: {best_candidate}")
ax.set_xlabel("posterior predictive SD")
ax.set_ylabel("absolute error")
fig.tight_layout()
fig_paths["posterior_sd_vs_abs_error"] = FIGURE_DIR / f"model__bayesian_posterior_scale_sd_vs_abs_error__{RUN_DATE}.png"
fig.savefig(fig_paths["posterior_sd_vs_abs_error"], dpi=160)
plt.close(fig)

well_best = well_diagnostics.loc[well_diagnostics["candidate"] == best_candidate].copy()
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(well_best["mean_residual"], bins=30, color="#72B7B2", edgecolor="white")
ax.axvline(0.0, color="black", linewidth=1)
ax.set_title(f"Well-Level Mean Residuals: {best_candidate}")
ax.set_xlabel("mean residual by well")
ax.set_ylabel("well count")
fig.tight_layout()
fig_paths["well_bias_distribution"] = FIGURE_DIR / f"model__bayesian_posterior_scale_well_bias_distribution__{RUN_DATE}.png"
fig.savefig(fig_paths["well_bias_distribution"], dpi=160)
plt.close(fig)

{key: str(value) for key, value in fig_paths.items()}


{'rmse_comparison': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\model__bayesian_posterior_scale_rmse_comparison__2026-06-22.png',
 'coverage_rmse_by_boundary': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\model__bayesian_posterior_scale_coverage_rmse_by_boundary__2026-06-22.png',
 'residual_bias_by_progress': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\model__bayesian_posterior_scale_residual_bias_by_progress__2026-06-22.png',
 'posterior_sd_vs_abs_error': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\model__bayesian_posterior_scale_sd_vs_abs_error__2026-06-22.png',
 'well_bias_distribution': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\model__bayesian_posterior_scale_well_bias_distribution__2026-06-22.png'}

## 6. Findings And Run Bundle

In [7]:
"Write report, update comparison table, and create run bundle."

bayes_summary = summary.loc[summary["candidate"] != "endpoint_hold"].copy()
best = bayes_summary.iloc[0]
reference = summary.loc[summary["candidate"] == "fixed_t3_autodelta_reference"].iloc[0]
endpoint = summary.loc[summary["candidate"] == "endpoint_hold"].iloc[0]
previous_tuned_bayes_rmse = 14.844948333805148
huber_glm_rmse = 14.792842528391505
best_well = well_diagnostics.loc[well_diagnostics["candidate"] == best.candidate]
well_bias_abs_q90 = float(best_well["mean_residual"].abs().quantile(0.90))
coverage_gap = float(abs(best.coverage_80_mean - 0.80))

findings = pd.DataFrame([
    {"topic": "best_scale_candidate", "finding": f"Best targeted scale candidate is {best.candidate} with sampled grouped RMSE {best.rmse_mean:.4f} and 80% coverage {best.coverage_80_mean:.3f}.", "modeling_implication": "Use this result to decide whether heteroskedastic scale structure improves the Bayesian linear baseline enough to keep tuning."},
    {"topic": "fixed_reference_comparison", "finding": f"The fixed Student-t reference scored RMSE {reference.rmse_mean:.4f}; best targeted scale delta is {best.rmse_mean - reference.rmse_mean:.4f}.", "modeling_implication": "If the delta is small or positive, move scale complexity into HSGP/hierarchical models rather than standalone Bayesian linear notebooks."},
    {"topic": "uncertainty_error_alignment", "finding": f"Best candidate mean Spearman correlation between posterior SD and absolute error is {best.sd_abs_error_corr_mean:.3f}.", "modeling_implication": "Low correlation means intervals are calibrated globally but weak at ranking difficult rows."},
    {"topic": "well_bias_signal", "finding": f"The 90th percentile absolute well-level mean residual for the best candidate is {well_bias_abs_q90:.2f} TVT units.", "modeling_implication": "Persistent well-level signed bias supports hierarchical well/intercept structure after the HSGP baseline."},
    {"topic": "hsgp_readiness", "finding": f"Best targeted scale RMSE vs Huber GLM local reference is {best.rmse_mean - huber_glm_rmse:.4f}, with coverage gap from nominal 80% of {coverage_gap:.3f}.", "modeling_implication": "The next Bayesian notebook should add smooth residual functions rather than only changing linear likelihood scale."},
])
findings_path = TABLE_DIR / f"model__bayesian_posterior_scale_findings__{RUN_DATE}.csv"
findings.to_csv(findings_path, index=False)

report_path = REPORT_DIR / f"model__bayesian_posterior_scale__{RUN_DATE}.md"
report_lines = ["# Bayesian Posterior Diagnostics And Targeted Scale", "", f"Date: {RUN_DATE}", "", "## Key Findings"]
for row in findings.itertuples(index=False):
    report_lines.extend([f"- **{row.topic}**: {row.finding}", f"  - Modeling implication: {row.modeling_implication}"])
report_lines.extend(["", "## Best Candidate", f"- Candidate: `{best.candidate}`", f"- Guide: `{best.guide}`", f"- Scale family: `{best.scale_family}`", f"- RMSE mean/std: {best.rmse_mean:.4f} / {best.rmse_std:.4f}", f"- 80% coverage/width: {best.coverage_80_mean:.3f} / {best.width_80_mean:.2f}", f"- Posterior SD vs absolute error Spearman: {best.sd_abs_error_corr_mean:.3f}", "", "## Artifacts", f"- Fold metrics: {fold_metrics_path}", f"- Summary: {summary_path}", f"- Posterior diagnostics: {posterior_path}", f"- Residual sample: {residual_path}", f"- Bin diagnostics: {bin_path}", f"- Well diagnostics: {well_path}", f"- Findings: {findings_path}"])
for name, fig_path in fig_paths.items():
    report_lines.append(f"- {name}: {fig_path}")
report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")

comparison_path = TABLE_DIR / "model_comparison.csv"
comparison = pd.read_csv(comparison_path)
run_comparison_row = pd.DataFrame([{"run_id": "pending_run_bundle", "model_family": "bayesian_posterior_scale", "model_name": str(best.candidate), "cv_rmse_mean": float(best.rmse_mean), "cv_rmse_std": float(best.rmse_std), "train_time_s": float(build_seconds + fit_seconds), "n_features": int(len(feature_columns) + fold_metrics.loc[fold_metrics["candidate"] == best.candidate, "n_sigma_features"].max()), "notes": "NumPyro/JAX posterior diagnostics with targeted heteroskedastic Student-t scale models"}])
comparison = comparison.loc[comparison["run_id"] != "pending_run_bundle"]
comparison = pd.concat([comparison, run_comparison_row], ignore_index=True)
comparison.to_csv(comparison_path, index=False)

config = {"task": "bayesian_posterior_diagnostics_scale", "seed": SEED, "n_splits": N_SPLITS, "max_target_rows_per_well": MAX_TARGET_ROWS_PER_WELL, "max_train_rows_per_fold": MAX_TRAIN_ROWS_PER_FOLD, "svi_steps": SVI_STEPS, "learning_rate": LEARNING_RATE, "posterior_samples": POSTERIOR_SAMPLES, "best_candidate": str(best.candidate), "note": "Posterior predictive diagnostics and targeted heteroskedastic scale before HSGP."}
with RunBundle(RUN_TAG, config, seed=SEED) as rb:
    rb.add_input(str(TRAIN_DIR))
    rb.add_metric("sample_rows", float(len(feature_frame)))
    rb.add_metric("best_rmse", float(best.rmse_mean))
    rb.add_metric("best_rmse_std", float(best.rmse_std))
    rb.add_metric("best_coverage_80", float(best.coverage_80_mean))
    rb.add_metric("best_width_80", float(best.width_80_mean))
    rb.add_metric("best_sd_abs_error_corr", float(best.sd_abs_error_corr_mean))
    rb.add_metric("fixed_reference_rmse", float(reference.rmse_mean))
    rb.add_metric("endpoint_hold_rmse", float(endpoint.rmse_mean))
    rb.add_metric("best_minus_previous_tuned_bayes", float(best.rmse_mean - previous_tuned_bayes_rmse))
    rb.add_metric("best_minus_huber_glm", float(best.rmse_mean - huber_glm_rmse))
    rb.add_metric("well_bias_abs_q90", float(well_bias_abs_q90))
    for output_path in [fold_metrics_path, summary_path, posterior_path, residual_path, bin_path, well_path, findings_path, report_path, comparison_path]:
        rb.add_output(str(output_path))
    for fig_path in fig_paths.values():
        rb.add_output(str(fig_path))
    rb.set_notes("Posterior predictive diagnostics and targeted scale-model sweep completed before HSGP work.")
run_id = rb.run_id
comparison.loc[comparison["run_id"] == "pending_run_bundle", "run_id"] = run_id
comparison.to_csv(comparison_path, index=False)

print({"run_id": run_id, "best_candidate": best.candidate, "best_rmse": round(float(best.rmse_mean), 4), "coverage_80": round(float(best.coverage_80_mean), 3), "sd_abs_error_corr": round(float(best.sd_abs_error_corr_mean), 3), "best_minus_reference": round(float(best.rmse_mean - reference.rmse_mean), 4)})
findings


{'run_id': '2026-06-22__1254__bayesian_posterior_diagnostics_scale__6644c20', 'best_candidate': 'hetero_boundary_progress_autodelta', 'best_rmse': 14.8626, 'coverage_80': 0.78, 'sd_abs_error_corr': 0.315, 'best_minus_reference': -0.0503}


,topic,finding,modeling_implication
0,best_scale_candidate,Best targeted scale candidate is hetero_bounda...,Use this result to decide whether heteroskedas...
1,fixed_reference_comparison,The fixed Student-t reference scored RMSE 14.9...,"If the delta is small or positive, move scale ..."
2,uncertainty_error_alignment,Best candidate mean Spearman correlation betwe...,Low correlation means intervals are calibrated...
3,well_bias_signal,The 90th percentile absolute well-level mean r...,Persistent well-level signed bias supports hie...
4,hsgp_readiness,Best targeted scale RMSE vs Huber GLM local re...,The next Bayesian notebook should add smooth r...


## Results

Run all cells to refresh the posterior predictive diagnostics and targeted heteroskedastic scale comparison.
